# 04 — Criteria-Version Reconciliation, from scratch

Companion notebook to `../07-model-and-criteria-staleness.md`.

This notebook implements the dual-tagging and reconciliation design Chapter 7 proposes for tracking
**two independent staleness axes**:

1. Whether the **model version** that screened an abstract is current.
2. Whether the **criteria version** an abstract was screened against is current.

These drift independently — a criteria refinement and a model retrain are two different events on two
different timelines — so this notebook tags every screening decision with both identifiers, and
implements two separate reconciliation checks, showing they can each flag a different subset of
decisions for the same review at the same time.

Fully offline: standard library only, no model calls, no API keys.

## 1. Modeling a screening decision with both version tags

Every screening decision carries the `model_version` and `criteria_version` it was made under, stamped
at write time — never derived after the fact.

In [1]:
from dataclasses import dataclass

@dataclass
class ScreeningDecision:
    abstract_id: str
    review_id: str
    model_version: str
    criteria_version: int
    verdict: str                          # "include" | "exclude"
    review_status: str = "auto_screened"  # "auto_screened" | "human_confirmed" | "needs_rescreen"


# Three abstracts screened under criteria v1, model v2026.01, for the same review
decisions = [
    ScreeningDecision("A001", "review-42", "deepseek-v3-oncology-2026.01", 1, "include"),
    ScreeningDecision("A002", "review-42", "deepseek-v3-oncology-2026.01", 1, "exclude"),
    ScreeningDecision("A003", "review-42", "deepseek-v3-oncology-2026.01", 1, "include"),
]

for d in decisions:
    print(d)


ScreeningDecision(abstract_id='A001', review_id='review-42', model_version='deepseek-v3-oncology-2026.01', criteria_version=1, verdict='include', review_status='auto_screened')
ScreeningDecision(abstract_id='A002', review_id='review-42', model_version='deepseek-v3-oncology-2026.01', criteria_version=1, verdict='exclude', review_status='auto_screened')
ScreeningDecision(abstract_id='A003', review_id='review-42', model_version='deepseek-v3-oncology-2026.01', criteria_version=1, verdict='include', review_status='auto_screened')


## 2. A criteria refinement lands mid-review

The review team refines the inclusion criteria (e.g., narrowing a date range or clarifying an
ambiguous case), bumping the review's `criteria_version`. Nothing about the model changes.

In [2]:
CURRENT_CRITERIA_VERSION = {"review-42": 1}
CURRENT_MODEL_VERSION = {"review-42": "deepseek-v3-oncology-2026.01"}


def refine_criteria(review_id: str) -> int:
    CURRENT_CRITERIA_VERSION[review_id] += 1
    return CURRENT_CRITERIA_VERSION[review_id]


new_criteria_version = refine_criteria("review-42")
print(f"Criteria refined -> review-42 is now at criteria_version={new_criteria_version}")
print(f"Model version is unchanged: {CURRENT_MODEL_VERSION['review-42']}")


Criteria refined -> review-42 is now at criteria_version=2
Model version is unchanged: deepseek-v3-oncology-2026.01


## 3. Reconciliation pass #1: flag decisions stale relative to the criteria

Directly from Chapter 7, Part 4. Any decision whose `criteria_version` trails the review's current
version gets flagged `needs_rescreen` -- staleness becomes a visible, tracked state rather than a
silent gap.

In [3]:
def flag_stale_screening_decisions(review_id: str, current_criteria_version: int, decisions: list):
    """Any decision whose criteria_version trails the review's current criteria_version is
    stale relative to the CRITERIA axis specifically -- not necessarily wrong, but made against
    a now-superseded standard."""
    flagged = []
    for d in decisions:
        if d.review_id == review_id and d.criteria_version < current_criteria_version:
            d.review_status = "needs_rescreen"
            flagged.append(d.abstract_id)
    return flagged


criteria_flagged = flag_stale_screening_decisions("review-42", new_criteria_version, decisions)
print("Flagged for re-screening (criteria staleness):", criteria_flagged)

assert set(criteria_flagged) == {"A001", "A002", "A003"}
assert all(d.review_status == "needs_rescreen" for d in decisions)
print("\nOK: all three decisions, made under the now-superseded criteria_version=1, are correctly "
      "flagged needs_rescreen against the new criteria_version=2.")


Flagged for re-screening (criteria staleness): ['A001', 'A002', 'A003']

OK: all three decisions, made under the now-superseded criteria_version=1, are correctly flagged needs_rescreen against the new criteria_version=2.


## 4. Re-screening: resets to the SAME human-confirmation gate, never auto-approved

Re-screening a flagged abstract updates its `criteria_version` and verdict -- but resets its status to
`auto_screened`, not `human_confirmed`. A criteria update is never a fast path around human review
(mirroring course 11's chapter 06, Part 5 for its own version-bump discipline).

In [4]:
def rescreen(decision: ScreeningDecision, new_criteria_version: int, new_verdict: str):
    decision.criteria_version = new_criteria_version
    decision.verdict = new_verdict
    decision.review_status = "auto_screened"   # re-enters the same downstream human-confirmation gate


# Re-screen all three under the refined criteria -- A002 flips from exclude to include this time
rescreen(decisions[0], new_criteria_version, "include")
rescreen(decisions[1], new_criteria_version, "include")   # flips!
rescreen(decisions[2], new_criteria_version, "include")

for d in decisions:
    print(d)

assert all(d.criteria_version == new_criteria_version for d in decisions)
assert all(d.review_status == "auto_screened" for d in decisions)
print("\nOK: every previously-stale decision is re-screened against the current criteria and reset "
      "to auto_screened (NOT human_confirmed) -- it re-enters the same review gate a first-time "
      "screening decision would, exactly as Chapter 7 specifies.")


ScreeningDecision(abstract_id='A001', review_id='review-42', model_version='deepseek-v3-oncology-2026.01', criteria_version=2, verdict='include', review_status='auto_screened')
ScreeningDecision(abstract_id='A002', review_id='review-42', model_version='deepseek-v3-oncology-2026.01', criteria_version=2, verdict='include', review_status='auto_screened')
ScreeningDecision(abstract_id='A003', review_id='review-42', model_version='deepseek-v3-oncology-2026.01', criteria_version=2, verdict='include', review_status='auto_screened')

OK: every previously-stale decision is re-screened against the current criteria and reset to auto_screened (NOT human_confirmed) -- it re-enters the same review gate a first-time screening decision would, exactly as Chapter 7 specifies.


## 5. Reconciliation pass #2: model-version drift is tracked SEPARATELY

Now simulate the other axis: the therapeutic-area adapter gets retrained (Chapter 5's retraining
loop), producing a new `model_version` -- an entirely separate event from the criteria refinement
above, on its own timeline. The point of this section: model-version staleness and criteria-version
staleness are checked independently, and flagging one says nothing about the other.

In [5]:
def flag_stale_model_version(review_id: str, current_model_version: str, decisions: list):
    """Any decision whose model_version differs from the review's current model_version is
    stale relative to the MODEL axis specifically -- independent of whatever criteria_version
    it carries."""
    flagged = []
    for d in decisions:
        if d.review_id == review_id and d.model_version != current_model_version:
            d.review_status = "needs_rescreen"
            flagged.append(d.abstract_id)
    return flagged


# The oncology adapter is retrained -> a new model_version, criteria are untouched
new_model_version = "deepseek-v3-oncology-2026.04"
CURRENT_MODEL_VERSION["review-42"] = new_model_version

model_flagged = flag_stale_model_version("review-42", new_model_version, decisions)
print("Flagged for re-screening (model-version staleness):", model_flagged)

# At this point every decision is current on criteria_version (Section 4) but stale on
# model_version -- proving the two axes are genuinely independent, not two names for one thing.
assert set(model_flagged) == {"A001", "A002", "A003"}
assert all(d.criteria_version == new_criteria_version for d in decisions), \
    "criteria_version should NOT have changed just because model_version drifted"
print("\nOK: all three decisions are current on criteria_version but flagged stale on "
      "model_version -- the two axes drift and get caught independently, exactly the distinction "
      "Chapter 7, Part 3 insists on not conflating.")


Flagged for re-screening (model-version staleness): ['A001', 'A002', 'A003']

OK: all three decisions are current on criteria_version but flagged stale on model_version -- the two axes drift and get caught independently, exactly the distinction Chapter 7, Part 3 insists on not conflating.


## 6. Tying it back

- Every screening decision carries **both** a `model_version` and a `criteria_version`, stamped at
  the moment it's made (Section 1) -- the precondition for either kind of staleness being a
  checkable question at all.
- **`flag_stale_screening_decisions`** (Section 3) and **`flag_stale_model_version`** (Section 5) are
  two separate reconciliation mechanisms over the same records, and Section 5's assertions show they
  really are independent: a review can be fully current on one axis and stale on the other at the
  same time.
- **Re-screening always resets to `auto_screened`, never `human_confirmed`** (Section 4) -- a version
  bump, on either axis, is never a fast path around the human sign-off gate Chapter 6 and Chapter 8
  describe.
- This is the concrete mechanism that answers "how do you know what standard produced this screening
  decision" (Chapter 2's regulatory-auditability argument) precisely, on both axes, rather than with
  a shrug.